In [1]:
print('Đang khai báo thư viện')

import pandas as pd
import numpy as np
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

print('Khai báo thư viện thành công')

Đang khai báo thư viện
Khai báo thư viện thành công


In [2]:
print('Khai báo dữ liệu')

train_path = "data_train_cleaned.csv"

train_path = os.path.join("..", "Notebooks", "DataPreprocessing", "data_train_cleaned.csv")
val_path = os.path.join("..", "Notebooks", "DataPreprocessing", "validation_cleaned.csv")

df_train = pd.read_csv(train_path)
df_val = pd.read_csv(val_path)

print("THÔNG TIN DỮ LIỆU")

print(f"Train shape: {df_train.shape}")
print(f"Validation shape: {df_val.shape}")
print(f"\nTrain columns:\n{df_train.columns.tolist()}")
print(f"\nClass distribution trong train:\n{df_train['Class'].value_counts().sort_index()}")
print(f"\nClass distribution trong validation:\n{df_val['Class'].value_counts().sort_index()}")

Khai báo dữ liệu
THÔNG TIN DỮ LIỆU
Train shape: (2880, 17)
Validation shape: (11516, 20)

Train columns:
['Popularity', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms', 'time_signature', 'Class', 'key_sin', 'key_cos']

Class distribution trong train:
Class
0     100
1     220
2     204
3      64
4      62
5     231
6     414
7      92
8     297
9     404
10    792
Name: count, dtype: int64

Class distribution trong validation:
Class
0      400
1      878
2      814
3      258
4      248
5      926
6     1655
7      369
8     1186
9     1615
10    3167
Name: count, dtype: int64


In [3]:
# Bỏ các cột không cần thiết (nếu có)
drop_cols = ['Id', 'Artist Name', 'Track Name']
existing_drop_train = [col for col in drop_cols if col in df_train.columns]
existing_drop_val = [col for col in drop_cols if col in df_val.columns]

X_train = df_train.drop(columns=['Class'] + existing_drop_train)
y_train = df_train['Class']

X_val = df_val.drop(columns=['Class'] + existing_drop_val)
y_val = df_val['Class']

print(f"\nX_train shape: {X_train.shape}")
print(f"X_val shape: {X_val.shape}")


X_train shape: (2880, 16)
X_val shape: (11516, 16)


In [4]:
print(f"Số cột train: {X_train.shape[1]}")
print(f"Số cột validation: {X_val.shape[1]}")

if X_train.shape[1] != X_val.shape[1]:
    print("\nSố cột ở val và train đang sai khác")
    missing_in_val = set(X_train.columns) - set(X_val.columns)
    missing_in_train = set(X_val.columns) - set(X_train.columns)
    
    if missing_in_val:
        print(f"Cột có trong train nhưng không trong val: {missing_in_val}")
        for col in missing_in_val:
            X_val[col] = 0  
    
    if missing_in_train:
        print(f"Cột có trong val nhưng không trong train: {missing_in_train}")
    
    X_val = X_val[X_train.columns]
    print(f"\nĐã căn chỉnh xong. Số cột val: {X_val.shape[1]}")
else:
    print("Số cột khớp nhau!")


Số cột train: 16
Số cột validation: 16
Số cột khớp nhau!


In [ ]:
rf_tuned = RandomForestClassifier(
    n_estimators=200,
    max_depth=30,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features='sqrt',
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
)
rf_tuned.fit(X_train, y_train)
y_pred_tuned = rf_tuned.predict(X_val)
y_train_pred_tuned = rf_tuned.predict(X_train)

In [ ]:
# ============================================
# RANDOM FOREST - GIẢM COMPLEXITY
# ============================================
rf_configs = {
    'WEAK': {
        'n_estimators': 50,
        'max_depth': 10,
        'min_samples_split': 10,
        'min_samples_leaf': 4,
        'max_features': 'sqrt'
    },
    'MEDIUM': {
        'n_estimators': 100,
        'max_depth': 15,
        'min_samples_split': 5,
        'min_samples_leaf': 2,
        'max_features': 'sqrt'
    },
    'REGULARIZED': {
        'n_estimators': 80,
        'max_depth': 8,
        'min_samples_split': 20,
        'min_samples_leaf': 10,
        'max_features': 'log2'
    }
}

results = {}

# Train các config mới
for name, params in rf_configs.items():
    rf = RandomForestClassifier(
        n_estimators=params['n_estimators'],
        max_depth=params['max_depth'],
        min_samples_split=params['min_samples_split'],
        min_samples_leaf=params['min_samples_leaf'],
        max_features=params['max_features'],
        random_state=42,
        class_weight='balanced',
        n_jobs=-1
    )
    rf.fit(X_train, y_train)
    y_train_pred = rf.predict(X_train)
    y_val_pred = rf.predict(X_val)
    
    results[name] = {
        'model': rf,
        'train_acc': accuracy_score(y_train, y_train_pred),
        'val_acc': accuracy_score(y_val, y_val_pred),
        'gap': accuracy_score(y_train, y_train_pred) - accuracy_score(y_val, y_val_pred),
        'y_pred': y_val_pred,
        'params': params
    }

In [ ]:


print("=" * 60)
print("THỬ CÁC CẤU HÌNH KHÁC NHAU")
print("=" * 60)

for name, params in rf_configs.items():
    rf = RandomForestClassifier(
        n_estimators=params['n_estimators'],
        max_depth=params['max_depth'],
        min_samples_split=params['min_samples_split'],
        min_samples_leaf=params['min_samples_leaf'],
        max_features=params['max_features'],
        random_state=42,
        class_weight='balanced',
        n_jobs=-1
    )
    
    rf.fit(X_train, y_train)
    y_train_pred = rf.predict(X_train)
    y_val_pred = rf.predict(X_val)
    
    train_acc = accuracy_score(y_train, y_train_pred)
    val_acc = accuracy_score(y_val, y_val_pred)
    gap = train_acc - val_acc
    
    print(f"\n--- {name.upper()} ---")
    print(f"Params: {params}")
    print(f"Train Accuracy: {train_acc:.4f}")
    print(f"Validation Accuracy: {val_acc:.4f}")
    print(f"Gap: {gap:.4f}")
    
    if gap < 0.2:
        print("✅ Model khả quan!")
        best_rf = rf
        best_val_acc = val_acc

print(f"\n✅ Best validation accuracy: {best_val_acc:.4f}")

In [8]:
print("\n" + "=" * 60)
print("RANDOM FOREST - TUNING (RandomizedSearchCV)")
print("=" * 60)

from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'n_estimators': [50, 100, 200, 300],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None]
}

rf_tuned = RandomForestClassifier(
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
)

random_search = RandomizedSearchCV(
    rf_tuned,
    param_distributions=param_dist,
    n_iter=20,  # Thử 20 bộ tham số
    cv=5,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

print("Đang tìm tham số tối ưu... (có thể hơi lâu)")
random_search.fit(X_train, y_train)

print(f"\nBest parameters: {random_search.best_params_}")
print(f"Best cross-validation score: {random_search.best_score_:.4f}")

rf_best = random_search.best_estimator_
y_pred_tuned = rf_best.predict(X_val)

print(f"\nValidation Accuracy sau tuning: {accuracy_score(y_val, y_pred_tuned):.4f}")


RANDOM FOREST - TUNING (RandomizedSearchCV)
Đang tìm tham số tối ưu... (có thể hơi lâu)
Fitting 5 folds for each of 20 candidates, totalling 100 fits

Best parameters: {'n_estimators': 200, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': 20}
Best cross-validation score: 0.4465

Validation Accuracy sau tuning: 0.4480


In [9]:
print("SO SÁNH KẾT QUẢ\n")

print("\n--- RANDOM FOREST BASIC ---")
print(f"Accuracy: {accuracy_score(y_val, y_pred_basic):.4f}")
print(classification_report(y_val, y_pred_basic, zero_division=0))

print("\n--- RANDOM FOREST TUNED ---")
print(f"Accuracy: {accuracy_score(y_val, y_pred_tuned):.4f}")
print(classification_report(y_val, y_pred_tuned, zero_division=0))

SO SÁNH KẾT QUẢ


--- RANDOM FOREST BASIC ---


NameError: name 'y_pred_basic' is not defined

In [ ]:
# ============================================
# 7. CHECK OVERFITTING
# ============================================
y_train_pred = rf_best.predict(X_train)
train_acc = accuracy_score(y_train, y_train_pred)
val_acc = accuracy_score(y_val, y_pred_tuned)

print("\n" + "=" * 60)
print("KIỂM TRA OVERFITTING")
print("=" * 60)
print(f"Train Accuracy: {train_acc:.4f}")
print(f"Validation Accuracy: {val_acc:.4f}")
print(f"Overfitting gap: {train_acc - val_acc:.4f}")

if train_acc - val_acc > 0.1:
    print("⚠️ Có dấu hiệu overfitting! Cần tăng regularization hoặc giảm độ phức tạp model.")
else:
    print("✅ Model generalize tốt!")

In [ ]:
# ============================================
# 8. CONFUSION MATRIX
# ============================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Basic model
cm_basic = confusion_matrix(y_val, y_pred_basic)
sns.heatmap(cm_basic, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=sorted(y_val.unique()), 
            yticklabels=sorted(y_val.unique()))
axes[0].set_title('Random Forest Basic - Confusion Matrix', fontweight='bold')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# Tuned model
cm_tuned = confusion_matrix(y_val, y_pred_tuned)
sns.heatmap(cm_tuned, annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=sorted(y_val.unique()), 
            yticklabels=sorted(y_val.unique()))
axes[1].set_title('Random Forest Tuned - Confusion Matrix', fontweight='bold')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.suptitle('Confusion Matrix Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# 9. FEATURE IMPORTANCE
# ============================================
feat_imp = pd.Series(rf_best.feature_importances_, index=X_train.columns).sort_values(ascending=False)

plt.figure(figsize=(12, 6))
colors = plt.cm.viridis(np.linspace(0, 1, len(feat_imp)))
feat_imp.plot(kind='bar', color=colors, edgecolor='black')
plt.title('Feature Importance - Random Forest (Tuned)', fontweight='bold', fontsize=14)
plt.ylabel('Importance Score', fontsize=12)
plt.xlabel('Features', fontsize=12)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.tight_layout()
plt.show()

print("\n" + "=" * 60)
print("TOP 10 IMPORTANT FEATURES")
print("=" * 60)
for i, (feat, imp) in enumerate(feat_imp.head(10).items(), 1):
    print(f"{i:2d}. {feat:25s} : {imp:.4f}")

In [ ]:
# ============================================
# 10. LƯU MODEL (nếu muốn dùng sau)
# ============================================
import joblib

joblib.dump(rf_best, 'random_forest_best_model.pkl')
print("\n✅ Đã lưu model vào file: random_forest_best_model.pkl")